# Experiment: Cross-Method Tiny-p Study

Objective:
- Compare `iid`, `mcmc_is`, and `samc` on fixed exact scenarios.
- Track estimates and diagnostics at intermediate estimation points, not just at the final budget.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    MCMCWorkflowConfig,
    SAMCWorkflowConfig,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_selected_scenarios,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
)

pd.set_option("display.max_columns", 100)
project_root

PosixPath('/Users/noamchowers/Documents/University/Thesis/Code/MCMCIS')

## Configuration

`ESTIMATION_POINTS` controls the intermediate checkpoints.  
The largest checkpoint is used for the main boxplots; all checkpoints are used for convergence diagnostics.  
In the cross-method notebook these are total budgets. For MCMC-IS, a fixed beta-selection budget is deducted first, and the production chain uses the remaining budget.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "cross_method_notebook"

SCENARIO_KEYS_TO_RUN = [
    "hypergeom_1e7",
    "gwas_additive_score_n40",
    "linear_stat_dp_n40",
    "bruteforce_welch_nonextreme_n22",
]

ESTIMATION_POINTS = (333_000, 1_000_000, 2_500_000, 5_000_000, 10_000_000) if not FAST_MODE else (2_000, 10_000, 20_000)
N_REPEATS = 7 if not FAST_MODE else 2
N_JOBS = min(N_REPEATS, os.cpu_count() or 1)
MIN_TAIL_STATES = 2
BASE_SEED = 12_345

cross_cfg = CrossMethodStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=N_REPEATS,
    base_seed=BASE_SEED,
    iid_density_samples=120_000 if not FAST_MODE else 10_000,
    min_tail_states=MIN_TAIL_STATES,
    n_jobs=N_JOBS,
)
mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=20_000 if not FAST_MODE else 1_000,
    tune_steps=2_000 if not FAST_MODE else 1_000,
    local_scan_total_steps=40_000 if not FAST_MODE else 4_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_fraction=0.1,
)
samc_cfg = SAMCWorkflowConfig(
    n_bins=50,
    t0=1_000.0,
    trace_every=200 if not FAST_MODE else 50,
    lambda_min_pilot=10_000 if not FAST_MODE else 500,
    proposal_fraction=0.1,
)

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_KEYS_TO_RUN": SCENARIO_KEYS_TO_RUN,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "N_REPEATS": N_REPEATS,
    "N_JOBS": N_JOBS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
}, indent=2))


{
  "FAST_MODE": false,
  "SCENARIO_KEYS_TO_RUN": [
    "hypergeom_1e7",
    "gwas_additive_score_n40",
    "linear_stat_dp_n40",
    "bruteforce_welch_nonextreme_n22"
  ],
  "ESTIMATION_POINTS": [
    250000,
    1000000,
    5000000
  ],
  "N_REPEATS": 5,
  "N_JOBS": 5,
  "SAVE_OUTPUTS": true
}


## Load Scenarios

In [3]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_TO_RUN,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "cross_method") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
    }
    for s in scenarios
])

,scenario,exact_p,tail_hits,n_perm,exact_method
0,hypergeom_1e7,3.854286e-07,53130,137846528820,Fisher exact test (2x2; hypergeometric tail)
1,gwas_additive_score_n40,9.121811e-07,125741,137846528820,LinearStatisticDPSolver
2,linear_stat_dp_n40,8.124978e-09,1120,137846528820,LinearStatisticDPSolver
3,bruteforce_welch_nonextreme_n22,1.559328e-05,11,705432,BruteForceExactSolver


## Run Cross-Method Study

For each scenario:
- build one MCMC-IS beta workflow,
- evaluate all methods at every checkpoint in `ESTIMATION_POINTS`,
- save max-budget and convergence plots.

In [4]:
cross_results = {}

for scenario in scenarios:
    print(f"Running {scenario.key} | exact p={scenario.exact_p:.3e}")
    study = run_cross_method_study(
        scenario,
        cross_cfg=cross_cfg,
        mcmc_cfg=mcmc_cfg,
        samc_cfg=samc_cfg,
    )
    cross_results[scenario.key] = study

    if SAVE_OUTPUTS and run_dir is not None:
        save_cross_method_outputs(
            scenario,
            study,
            output_dir=run_dir / scenario.key,
            cross_cfg=cross_cfg,
            mcmc_cfg=mcmc_cfg,
            samc_cfg=samc_cfg,
        )

    print(json.dumps({
        "scenario": scenario.key,
        "mcmc_beta_selection_budget": study["mcmc_beta_selection_budget"],
        "beta_used": study["beta_workflow"]["beta_used"],
    }, indent=2))
    summary_df = pd.DataFrame(study["summary"]).sort_values(["checkpoint", "method"])
    display(summary_df[[
        "checkpoint",
        "method",
        "mean_estimate",
        "rmse",
        "mean_variance_estimate",
        "mean_eval_incl_tuning",
        "mean_q_tilt_tail_share",
        "mean_ess",
        "mean_zero_rate",
        "mean_samc_max_rel_freq_error",
    ]])

Running hypergeom_1e7 | exact p=3.854e-07


ValueError: Cross-method estimation_points must all exceed the fixed MCMC-IS beta-selection budget. Received estimation_points=[250000, 1000000, 5000000], beta_selection_budget=528019.

## Review Saved Figures

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    for scenario in scenarios:
        scenario_dir = run_dir / scenario.key
        print(f"\n{scenario.key}")
        display(Image(filename=str(scenario_dir / "cross_method_max_budget.png")))
        display(Image(filename=str(scenario_dir / "cross_method_convergence.png")))
        display(Image(filename=str(scenario_dir / "cross_method_diagnostics.png")))
        display(Image(filename=str(scenario_dir / "iid_density.png")))
else:
    print("SAVE_OUTPUTS=False, so no saved figures to display.")

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_SCENARIO_DIR = None
# # Example:
# # RELOAD_SCENARIO_DIR = project_root / "results" / "cross_method_notebook" / "20260306_120000_cross_method" / "gwas_additive_score_n40"

# if RELOAD_SCENARIO_DIR is not None:
#     saved = load_cross_method_saved_output(RELOAD_SCENARIO_DIR)
#     print(json.dumps({
#         "scenario": saved["metadata"]["scenario"],
#         "exact_p": saved["metadata"]["exact_p"],
#         "mcmc_beta_selection_budget": saved["metadata"]["beta_workflow"]["beta_selection_eval_total"],
#     }, indent=2))
#     regen = regenerate_cross_method_plots_from_saved(RELOAD_SCENARIO_DIR)
#     for name, path in regen.items():
#         print(name, path)
#         display(Image(filename=str(path)))
# else:
#     print("Set RELOAD_SCENARIO_DIR to a saved scenario directory to regenerate plots from disk only.")
